# Explainability & Credit Risk Scoring

This notebook explains the predictions of the trained XGBoost credit risk model using SHAP (SHapley Additive exPlanations) and converts probabilities into business-friendly credit risk scores.

**Objective:**
- Provide global and local model explainability
- Justify loan decisions
- Generate interpretable credit risk scores

## 1. Load Trained Pipeline

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

# Load trained pipeline (preprocessing + model)
pipeline = joblib.load('models/credit_risk_pipeline.pkl')

The pipeline contains both the preprocessing steps and the trained XGBoost model, ensuring consistency between training and inference.

## 2. Prepare Data for Explainability

In [ ]:
# Load dataset (same dataset used during training)
df = pd.read_csv('data/credit_risk.csv')

target = 'loan_status'
X = df.drop(columns=[target])
y = df[target]

# Use a subset for SHAP to reduce computation
X_sample = X.sample(1000, random_state=42)

## 3. SHAP Explainer Setup

In [ ]:
# Extract trained model from pipeline
model = pipeline.named_steps['model']
preprocessor = pipeline.named_steps['preprocessor']

# Transform data
X_sample_processed = preprocessor.transform(X_sample)

# Initialize SHAP explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample_processed)

SHAP values represent the contribution of each feature to the prediction. Positive values increase default risk, while negative values decrease it.

## 4. Global Explainability

In [ ]:
shap.summary_plot(shap_values, X_sample_processed)

In [ ]:
shap.summary_plot(shap_values, X_sample_processed, plot_type='bar')

**Insight:** Loan burden, interest rate, loan grade, and historical defaults are the most influential drivers of credit risk.

## 5. Local Explainability (Single Prediction)

In [ ]:
# Select a single observation
idx = 0
X_single = X_sample.iloc[[idx]]
X_single_processed = preprocessor.transform(X_single)

shap_value_single = explainer.shap_values(X_single_processed)

shap.force_plot(
    explainer.expected_value,
    shap_value_single,
    X_single_processed,
    matplotlib=True
)

This local explanation shows how individual features influenced the risk prediction for a specific applicant.

## 6. Credit Risk Score Generation

In [ ]:
def credit_risk_score(prob_default):
    return int(100 - prob_default * 100)

def risk_category(score):
    if score >= 80:
        return 'Low Risk'
    elif score >= 60:
        return 'Medium Risk'
    else:
        return 'High Risk'

In [ ]:
# Apply risk scoring to sample predictions
probs = pipeline.predict_proba(X_sample)[:,1]

results = X_sample.copy()
results['prob_default'] = probs
results['risk_score'] = results['prob_default'].apply(credit_risk_score)
results['risk_category'] = results['risk_score'].apply(risk_category)

results.head()

## 7. Explainability Summary

- SHAP confirms that the model aligns with financial intuition
- Historical defaults and loan burden are the strongest risk indicators
- Credit risk scores provide an interpretable decision framework

**Conclusion:** The explainability layer ensures regulatory compliance, model transparency, and business trust.